# 06 - CatBoost (TUNED)

## 1. What this model actually does
CatBoost is **gradient boosting on decision trees**: it builds many small trees one after another, where each new tree is trained to correct the errors the current ensemble still makes. Predictions are the sum of all trees' contributions, each scaled down by the `learning_rate`. Two things make CatBoost special:
- **Ordered boosting** - a clever training scheme that reduces a subtle form of overfitting ("target leakage") that ordinary boosting suffers from, so its defaults are unusually strong.
- **Symmetric (oblivious) trees** - every node at the same depth uses the same split, which acts as built-in regularization and makes prediction very fast.

Key knobs we tune:
- **`depth`** - tree depth. Deeper trees model higher-order feature interactions but can overfit. (4-10 typical.)
- **`learning_rate`** - size of each tree's correction. Lower = more careful, needs more `iterations`.
- **`iterations`** - number of trees (boosting rounds).
- **`l2_leaf_reg`** - L2 penalty on leaf values; higher = stronger regularization.

## 2. Why it fits THIS dataset well
- **Native NaN handling:** we feed the raw 50%-missing `eog_burst_index` with NO imputation - CatBoost learns the best direction for missing values at each split. This avoids inventing values, a real advantage on this dataset.
- Scale-invariant and robust to the strong feature correlations (up to 0.82) we found in EDA.
- Captures the nonlinear, overlapping class boundaries that linear models cannot.
- Strong, low-variance defaults - it was already the best tree model (~0.821 untuned).

## 3. Its costs / weaknesses here
- Its headline feature (native handling of *categorical* columns) is irrelevant here - all 21 features are numeric.
- Training is heavier than HistGBM, so a full grid search is slow (see the timing note on the search cell).
- Already near its performance plateau on this data, so tuning yields only small gains - the value of the search is *confirming* the best config, not a big jump.

## 4. What changed vs the baseline
Baseline: `iterations=600, depth=6, learning_rate=0.05` -> CV macro-F1 **0.821**. A quick scan found `depth=7, learning_rate=0.04, l2_leaf_reg=3` slightly better (~0.822). The grid search cell below explores a wider space; whatever it finds as best is what the final submission uses, so re-running it cannot lower performance.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

# This notebook lives in day9/models/ ; the data is one folder up in day9/
TRAIN_PATH, TEST_PATH, OUT_DIR = '../train.csv', '../test.csv', '../outputs'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# Every column except the row id and the label is a predictor (21 physiological signals).
FEATURES = [c for c in train.columns if c not in ('id', 'sleep_stage')]
TARGET   = 'sleep_stage'
y = train[TARGET]
print('train', train.shape, '| test', test.shape, '| #features', len(FEATURES))
print('class balance:', dict(y.value_counts().sort_index()))
train.head()

train (9000, 23) | test (5000, 22) | #features 21
class balance: {0: np.int64(2001), 1: np.int64(2442), 2: np.int64(2237), 3: np.int64(2320)}


,id,eeg_delta_power,eeg_theta_power,eeg_alpha_power,eeg_sigma_power,eeg_beta_power,eeg_gamma_power,eeg_slow_osc_power,eeg_spectral_entropy,eeg_spindle_density,...,eog_movement_density,eog_amplitude,heart_rate_mean,heart_rate_variability,respiration_rate,respiration_variability,spo2_mean,body_movement_index,eog_burst_index,sleep_stage
0,0,-1.51474,1.40728,10.33510,-1.61350,3.73081,0.99850,1.85689,-3.24253,-1.27096,...,2.65567,1.98733,1.60184,-2.49794,-0.59521,1.71154,1.93342,1.57365,-1.36230,1
1,1,-0.28998,0.89706,1.62494,2.41580,-2.70265,-0.10131,-1.68955,0.01442,-2.87943,...,4.36423,0.09942,3.38567,-0.56386,2.16016,-4.32301,1.07270,-2.43903,-0.37271,2
2,2,3.35435,0.32987,-5.41547,2.38680,-2.90584,-2.93372,-3.11713,-0.04647,1.61782,...,-3.87561,-0.87681,-2.84480,5.08383,-4.60411,0.37967,-2.06913,2.67324,NaN,3
3,3,-1.44917,-0.04374,1.71560,-1.27770,-0.19007,2.21826,1.69621,0.39756,0.00534,...,1.41415,0.39275,0.55060,-2.12910,2.32790,0.78319,0.98233,1.53824,-0.25040,1
4,4,1.35898,-2.36720,-7.40779,5.31815,-2.55954,-5.13284,-5.26634,1.73985,1.04618,...,-0.55616,0.86588,-1.96343,4.30036,0.22130,-1.44020,1.35760,-3.07701,-1.04947,3


## 5. Preprocessing
**None.** CatBoost accepts NaN directly and is scale-invariant (trees split on thresholds, not distances), so we pass the raw feature matrix including the half-missing column. This is the cleanest possible treatment of the missing data - no values are imputed.

In [2]:
from catboost import CatBoostClassifier

def build_cat(**kw):
    params = dict(loss_function='MultiClass', random_state=42, verbose=0, thread_count=-1)
    params.update(kw)
    return CatBoostClassifier(**params)

# the current best-known config (the grid search below may refine it)
build_cat(iterations=600, depth=7, learning_rate=0.04, l2_leaf_reg=3)

CatBoostClassifier(depth=7, iterations=600, l2_leaf_reg=3, learning_rate=0.04, loss_function='MultiClass', random_state=42, verbose=0)

In [3]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# ---- The validation protocol (identical for every model so scores are comparable) ----
# The test set has no labels, so we cannot score on it locally. Instead we repeatedly
# hold out part of TRAIN and score there.
#   StratifiedKFold(5)  -> split the 9000 rows into 5 equal parts ("folds").
#   stratified          -> each fold keeps the same 22-27% class proportions.
#   shuffle + seed 42   -> reproducible split.
# cross_val_score runs 5x: train on 4 folds (~7200 rows), score the held-out fold
# (~1800 unseen rows), average the 5 scores.
#   scoring='f1_macro' -> competition metric: F1 per class, averaged with EQUAL weight.
cv = StratifiedKFold(5, shuffle=True, random_state=42)

def benchmark(model, X, label):
    s = cross_val_score(model, X, y, cv=cv, scoring='f1_macro')
    print(f'{label}: macro-F1 = {s.mean():.4f} +/- {s.std():.4f}')
    return s

## 6. Baseline vs current-best config (same 5-fold CV)
We feed `train[FEATURES]` directly - the NaNs are intentional and handled internally.

In [4]:
benchmark(build_cat(iterations=600, depth=6, learning_rate=0.05, l2_leaf_reg=3),
          train[FEATURES], 'CatBoost baseline (d6, lr.05)')
benchmark(build_cat(iterations=600, depth=7, learning_rate=0.04, l2_leaf_reg=3),
          train[FEATURES], 'CatBoost current-best (d7, lr.04)')

CatBoost baseline (d6, lr.05): macro-F1 = 0.8213 +/- 0.0064
CatBoost current-best (d7, lr.04): macro-F1 = 0.8216 +/- 0.0067


array([0.81487934, 0.81321891, 0.83147753, 0.82429412, 0.82422511])

## 7. The full tuning search  (SLOW - run this yourself)
GridSearchCV tries every `depth x learning_rate x l2_leaf_reg` combination with the SAME 5-fold CV and keeps the best by macro-F1.

**Runtime warning:** this grid is 3x3x3 = 27 configs x 5 folds = **135 CatBoost trainings of 800 trees each ~ 30-45 minutes** on a typical laptop. That is intentional - we are not cutting the search short. To do a faster pass, shrink the lists (e.g. `depth=[6,7]`, drop a learning_rate). `refit=True` means GridSearchCV automatically retrains the single best config on ALL training rows at the end, ready for prediction.

In [5]:
from sklearn.model_selection import GridSearchCV

grid = {
    'depth':         [6, 7, 8],
    'learning_rate': [0.03, 0.05, 0.08],
    'l2_leaf_reg':   [1, 3, 5],
}
gs = GridSearchCV(
    build_cat(iterations=800),   # fixed 800 trees; lower lr benefits from more trees
    grid, scoring='f1_macro', cv=cv, n_jobs=1,  # n_jobs=1: let CatBoost use all cores per fit
)
gs.fit(train[FEATURES], y)
print('best params:', gs.best_params_)
print('best macro-F1 = %.4f' % gs.best_score_)
pd.DataFrame(gs.cv_results_).sort_values('rank_test_score')[
    ['param_depth','param_learning_rate','param_l2_leaf_reg','mean_test_score','std_test_score']].head(8)

best params: {'depth': 7, 'l2_leaf_reg': 3, 'learning_rate': 0.05}
best macro-F1 = 0.8260


,param_depth,param_learning_rate,param_l2_leaf_reg,mean_test_score,std_test_score
13,7,0.05,3,0.826036,0.006028
16,7,0.05,5,0.825776,0.003018
10,7,0.05,1,0.824639,0.004670
19,8,0.05,1,0.824273,0.006136
17,7,0.08,5,0.824233,0.006395
1,6,0.05,1,0.824112,0.005274
25,8,0.05,5,0.824053,0.006129
11,7,0.08,1,0.823968,0.003782


## 8. Train final model on ALL data and write the submission
`GridSearchCV(refit=True)` already retrained the best configuration on every labelled row, so `gs.best_estimator_` is ready to predict. We use it directly - this guarantees the submission reflects the BEST config the search found (never a hand-picked compromise). Output: `outputs/catboost_submission.csv`.

In [6]:
import os
os.makedirs(OUT_DIR, exist_ok=True)

final_cat = gs.best_estimator_              # best config, already fit on all 9000 rows
pred = final_cat.predict(test[FEATURES])
submission = pd.DataFrame({'id': test['id'], 'sleep_stage': np.asarray(pred).astype(int).ravel()})
path = os.path.join(OUT_DIR, 'catboost_submission.csv')
submission.to_csv(path, index=False)
print('wrote', path, submission.shape)
print('predicted class distribution:', dict(submission.sleep_stage.value_counts().sort_index()))
submission.head()

wrote ../outputs/catboost_submission.csv (5000, 2)
predicted class distribution: {0: np.int64(1106), 1: np.int64(1308), 2: np.int64(1271), 3: np.int64(1315)}


,id,sleep_stage
0,9000,0
1,9001,3
2,9002,1
3,9003,2
4,9004,3


### (Optional) Skip the slow search and submit with the current-best config
If you don't want to wait for the grid search, run THIS cell instead of cells 7-8. It uses the known-good `depth=7, lr=0.04` config directly.

In [ ]:
# import os
# os.makedirs(OUT_DIR, exist_ok=True)
# final_cat = build_cat(iterations=600, depth=7, learning_rate=0.04, l2_leaf_reg=3)
# final_cat.fit(train[FEATURES], y)
# pred = final_cat.predict(test[FEATURES])
# submission = pd.DataFrame({'id': test['id'], 'sleep_stage': np.asarray(pred).astype(int).ravel()})
# submission.to_csv(os.path.join(OUT_DIR, 'catboost_submission.csv'), index=False)
# print('wrote quick submission', submission.shape)

## 9. How to push further later
- Use `eval_set` + `early_stopping_rounds` to let CatBoost pick the optimal number of trees automatically (often beats a fixed `iterations`).
- Widen the search: add `border_count`, `bagging_temperature`, `random_strength`.
- Inspect `final_cat.get_feature_importance()` and a confusion matrix; engineer features aimed at class 2 (the weakest class that caps macro-F1).
- Reserve CatBoost as one half of a future CatBoost + SVM ensemble (their errors differ).